### Gorgon

### Read data

In [ ]:
import pandas as pd
df_sppid_mtl=pd.read_excel("Smartplant_PID_Extracts_gor.xlsx", sheet_name="CVX_MTL")

### columns to keep

In [27]:
rename_map_sppid={'TAGID':'tagNumber', 'CATEGORY':'category', 'TYPE':'type', 'PID NUMBER':'pidNumber', 'PROJECT':'status'}
df_sppid_gor=df_sppid_mtl.rename(columns=rename_map_sppid)
#df_sppid_mtl.to_csv('SPPID_CVX_MTL.tsv', sep="\t", index=False)

# Upload sppidRecord node to DB

In [28]:
from neo4j import GraphDatabase
import pandas as pd

URI = os.getenv("NEO4J_URI")
USERNAME = os.getenv("NEO4J_USERNAME")
PASSWORD = os.getenv("NEO4J_PASSWORD")
DATABASE = os.getenv("NEO4J_DATABASE")

BATCH_SIZE = 5000   # you can try 5000–20000 depending on RAM

# IMPORTANT: Neo4j label names are case-sensitive.
# Set this to your actual tag label in Neo4j: "tag" or "Tag"
TAG_LABEL = "Tag"   # <-- change to "Tag" if your nodes are :Tag

SENTINELS = {"", "-", "--", "---", "unset", "*", "NaN", "nan", "None", "<NA>"}

# Create index to speed MATCH on tagNumber (doesn't change logic)
INDEX_QUERY = f"CREATE INDEX tag_tagNumber IF NOT EXISTS FOR (t:{TAG_LABEL}) ON (t.tagNumber)"


# Cypher: same logic, faster property setting
CYPHER = f"""
UNWIND $batch AS row
CREATE (s:sppidRecord)
SET s += row.props

WITH s, row
WHERE row.tagNumber IS NOT NULL

MATCH (t:{TAG_LABEL} {{tagNumber: row.tagNumber}})
MERGE (t)-[:PRESENT_IN]->(s)
"""


def ensure_index(driver):
    with driver.session(database=DATABASE) as session:
        session.run(INDEX_QUERY).consume()
        print(f"Ensured index: {INDEX_QUERY}")


def _clean_value(v):
   
    if v is None:
        return None
    if isinstance(v, str):
        v = v.strip()
        if v in SENTINELS:
            return None
        return v
    # handle pandas NaN
    try:
        # NaN != NaN in Python
        if v != v:
            return None
    except Exception:
        pass
    return v


def _row_payload(row_dict):
    """
    Build payload:
      - props: only valid properties + siteName="Gorgon"
      - tagNumber: separate for MATCH (same as original WHERE)
    """
    props = {}
    for k, v in row_dict.items():
        cv = _clean_value(v)
        if cv is not None:
            props[k] = cv

   
    props["siteName"] = "Gorgon"

    # Use cleaned tagNumber for the relationship logic
    tag_val = props.get("tagNumber", None)
    if tag_val is None:
        # still create s node (same as original), but relationship won't happen
        return {"props": props, "tagNumber": None}

    return {"props": props, "tagNumber": tag_val}


def upload_sppid_from_df(df_in: pd.DataFrame, batch_size=BATCH_SIZE):
    # Copy + same header cleaning you did
    df = df_in.copy()
    df.columns = (
        df.columns
          .str.strip()
          
    )

    total = len(df)
    print(f"Prepared {total} rows (DataFrame input). Uploading in batches of {batch_size}...")

    driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))
    try:
        ensure_index(driver)

        with driver.session(database=DATABASE) as session:
            processed = 0
            batch_num = 0

            # Stream batches directly from the DataFrame (faster, less memory)
            for start in range(0, total, batch_size):
                batch_num += 1
                batch_df = df.iloc[start:start + batch_size]

                # Convert only this batch to dicts
                raw_records = batch_df.to_dict(orient="records")

                # Build payloads (filtered props + tagNumber)
                batch_payload = [_row_payload(r) for r in raw_records]

                # Write transaction
                result = session.run(CYPHER, batch=batch_payload).consume()
                counters = result.counters

                processed += len(batch_payload)
                print(
                    f"[Batch {batch_num}] sent {len(batch_payload)} | "
                    
                )

        print("=====================================")
        print("Upload complete.")
        print("=====================================")

    finally:
        driver.close()


# ---- Example call (your DataFrame) ----
upload_sppid_from_df(df_sppid_gor)

Prepared 158787 rows (DataFrame input). Uploading in batches of 5000...
Ensured index: CREATE INDEX tag_tagNumber IF NOT EXISTS FOR (t:Tag) ON (t.tagNumber)
[Batch 1] sent 5000 | 
[Batch 2] sent 5000 | 
[Batch 3] sent 5000 | 
[Batch 4] sent 5000 | 
[Batch 5] sent 5000 | 
[Batch 6] sent 5000 | 
[Batch 7] sent 5000 | 
[Batch 8] sent 5000 | 
[Batch 9] sent 5000 | 
[Batch 10] sent 5000 | 
[Batch 11] sent 5000 | 
[Batch 12] sent 5000 | 
[Batch 13] sent 5000 | 
[Batch 14] sent 5000 | 
[Batch 15] sent 5000 | 
[Batch 16] sent 5000 | 
[Batch 17] sent 5000 | 
[Batch 18] sent 5000 | 
[Batch 19] sent 5000 | 
[Batch 20] sent 5000 | 
[Batch 21] sent 5000 | 
[Batch 22] sent 5000 | 
[Batch 23] sent 5000 | 
[Batch 24] sent 5000 | 
[Batch 25] sent 5000 | 
[Batch 26] sent 5000 | 
[Batch 27] sent 5000 | 
[Batch 28] sent 5000 | 
[Batch 29] sent 5000 | 
[Batch 30] sent 5000 | 
[Batch 31] sent 5000 | 
[Batch 32] sent 3787 | 
Upload complete.


# WSD


### Read data

In [29]:
import pandas as pd
df_sppid_WSD=pd.read_excel("Smartplant_PID_Extract_WSD.xlsx", sheet_name="CVX_MTL")

### Columns to keep

In [31]:
rename_map_sppid={'TAGID':'tagNumber', 'CATEGORY':'category', 'TYPE':'type', 'PID NUMBER':'pidNumber', 'PROJECT':'status'}
df_sppid_WSD=df_sppid_WSD.rename(columns=rename_map_sppid)

### Upload SppidRecord node

In [32]:
from neo4j import GraphDatabase
import pandas as pd



URI = os.getenv("NEO4J_URI")
USERNAME = os.getenv("NEO4J_USERNAME")
PASSWORD = os.getenv("NEO4J_PASSWORD")
DATABASE = os.getenv("NEO4J_DATABASE")

BATCH_SIZE = 5000   # you can try 5000–20000 depending on RAM

# IMPORTANT: Neo4j label names are case-sensitive.
# Set this to your actual tag label in Neo4j: "tag" or "Tag"
TAG_LABEL = "Tag"   # <-- change to "Tag" if your nodes are :Tag

SENTINELS = {"", "-", "--", "---", "unset", "*", "NaN", "nan", "None", "<NA>"}

# Create index to speed MATCH on tagNumber (doesn't change logic)
INDEX_QUERY = f"CREATE INDEX tag_tagNumber IF NOT EXISTS FOR (t:{TAG_LABEL}) ON (t.tagNumber)"


# Cypher: same logic, faster property setting
CYPHER = f"""
UNWIND $batch AS row
CREATE (s:sppidRecord)
SET s += row.props

WITH s, row
WHERE row.tagNumber IS NOT NULL

MATCH (t:{TAG_LABEL} {{tagNumber: row.tagNumber}})
MERGE (t)-[:PRESENT_IN]->(s)
"""


def ensure_index(driver):
    with driver.session(database=DATABASE) as session:
        session.run(INDEX_QUERY).consume()
        print(f"Ensured index: {INDEX_QUERY}")


def _clean_value(v):
   
    if v is None:
        return None
    if isinstance(v, str):
        v = v.strip()
        if v in SENTINELS:
            return None
        return v
    # handle pandas NaN
    try:
        # NaN != NaN in Python
        if v != v:
            return None
    except Exception:
        pass
    return v


def _row_payload(row_dict):
    """
    Build payload:
      - props: only valid properties + siteName="Gorgon"
      - tagNumber: separate for MATCH (same as original WHERE)
    """
    props = {}
    for k, v in row_dict.items():
        cv = _clean_value(v)
        if cv is not None:
            props[k] = cv

   
    props["siteName"] = "WSD"

    # Use cleaned tagNumber for the relationship logic
    tag_val = props.get("tagNumber", None)
    if tag_val is None:
        # still create s node (same as original), but relationship won't happen
        return {"props": props, "tagNumber": None}

    return {"props": props, "tagNumber": tag_val}


def upload_sppid_from_df(df_in: pd.DataFrame, batch_size=BATCH_SIZE):
    # Copy + same header cleaning you did
    df = df_in.copy()
    df.columns = (
        df.columns
          .str.strip()
          
    )

    total = len(df)
    print(f"Prepared {total} rows (DataFrame input). Uploading in batches of {batch_size}...")

    driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))
    try:
        ensure_index(driver)

        with driver.session(database=DATABASE) as session:
            processed = 0
            batch_num = 0

            # Stream batches directly from the DataFrame (faster, less memory)
            for start in range(0, total, batch_size):
                batch_num += 1
                batch_df = df.iloc[start:start + batch_size]

                # Convert only this batch to dicts
                raw_records = batch_df.to_dict(orient="records")

                # Build payloads (filtered props + tagNumber)
                batch_payload = [_row_payload(r) for r in raw_records]

                # Write transaction
                result = session.run(CYPHER, batch=batch_payload).consume()
                counters = result.counters

                processed += len(batch_payload)
                print(
                    f"[Batch {batch_num}] sent {len(batch_payload)} | "
                    
                )

        print("=====================================")
        print("Upload complete.")
        print("=====================================")

    finally:
        driver.close()


# ---- Example call (your DataFrame) ----
upload_sppid_from_df(df_sppid_WSD)

Prepared 182048 rows (DataFrame input). Uploading in batches of 5000...
Ensured index: CREATE INDEX tag_tagNumber IF NOT EXISTS FOR (t:Tag) ON (t.tagNumber)
[Batch 1] sent 5000 | 
[Batch 2] sent 5000 | 
[Batch 3] sent 5000 | 
[Batch 4] sent 5000 | 
[Batch 5] sent 5000 | 
[Batch 6] sent 5000 | 
[Batch 7] sent 5000 | 
[Batch 8] sent 5000 | 
[Batch 9] sent 5000 | 
[Batch 10] sent 5000 | 
[Batch 11] sent 5000 | 
[Batch 12] sent 5000 | 
[Batch 13] sent 5000 | 
[Batch 14] sent 5000 | 
[Batch 15] sent 5000 | 
[Batch 16] sent 5000 | 
[Batch 17] sent 5000 | 
[Batch 18] sent 5000 | 
[Batch 19] sent 5000 | 
[Batch 20] sent 5000 | 
[Batch 21] sent 5000 | 
[Batch 22] sent 5000 | 
[Batch 23] sent 5000 | 
[Batch 24] sent 5000 | 
[Batch 25] sent 5000 | 
[Batch 26] sent 5000 | 
[Batch 27] sent 5000 | 
[Batch 28] sent 5000 | 
[Batch 29] sent 5000 | 
[Batch 30] sent 5000 | 
[Batch 31] sent 5000 | 
[Batch 32] sent 5000 | 
[Batch 33] sent 5000 | 
[Batch 34] sent 5000 | 
[Batch 35] sent 5000 | 
[Batch 36] s